In [0]:
%reload_ext autoreload
%autoreload 2

In [0]:
from utils.logger import get_logger
from etl_config.gold_fact_config import FACT_CONFIG

logger = get_logger("gold_fact")

In [0]:
dbutils.widgets.text("fact_name", "")
fact_name = dbutils.widgets.get("fact_name")
# fact_name = 'fact_orders'

if not fact_name or fact_name not in FACT_CONFIG:
    raise ValueError(f"Invalid or missing fact_name widget: '{fact_name}'")

cfg = FACT_CONFIG[fact_name]
column_names = [c.name for c in cfg.columns]
logger.info(f"Starting Gold fact processing for: {fact_name}")

In [0]:
source_df = spark.sql(cfg.source_query).select(column_names)
source_df.createOrReplaceTempView(f"stg_{fact_name}")
source_count = source_df.count()
logger.info(f"'{fact_name}': source query returned {source_count} rows")

In [0]:
full_table_name = cfg.target_table

merge_condition = " and ".join([f"target.{k} = source.{k}" for k in cfg.key_columns])

non_key_columns = [c for c in column_names if c not in cfg.key_columns]
update_set = ", ".join([f"target.{c} = source.{c}" for c in non_key_columns])
insert_cols = ", ".join(column_names)
insert_vals = ", ".join([f"source.{c}" for c in column_names])

spark.sql(f"""
    merge into {full_table_name} as target
    using stg_{fact_name} as source
    on {merge_condition}
    when matched then
        update set {update_set}
    when not matched then
        insert ({insert_cols}) values ({insert_vals})
""")

logger.info(f"'{fact_name}': MERGE complete into {full_table_name}")

In [0]:
merge_metrics = spark.sql(f"describe history {full_table_name} limit 1") \
    .select("operationMetrics").collect()[0]["operationMetrics"]
inserted = int(merge_metrics.get("numTargetRowsInserted", 0))
updated = int(merge_metrics.get("numTargetRowsUpdated", 0))

total_count = spark.table(full_table_name).count()

logger.info(f"=== Gold Fact Summary: {fact_name} ===")
logger.info(f"  Source rows (from Silver):   {source_count}")
logger.info(f"  Inserted:                    {inserted}")
logger.info(f"  Updated:                     {updated}")
logger.info(f"  Total rows in table:         {total_count}")
logger.info("======================================")

logger.info(f"'{fact_name}': Gold fact processing complete.")